In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("test")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/ramadhan/dev/sandbox/dez/batch-processing/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/27 09:51:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
import glob

paths = glob.glob("../data/pq/green/*/*")

df_green = spark.read.parquet(*paths)

print(df_green.count())

8348567


In [5]:
# Alternatively
df_green = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet("../data/pq/green")
)

print(df_green.count())

8348567


In [6]:
df_green.rdd.take(5)

[Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2019, 1, 29, 15, 36, 35), lpep_dropoff_datetime=datetime.datetime(2019, 1, 29, 15, 45, 15), store_and_fwd_flag='N', RatecodeID=1, PULocationID=40, DOLocationID=181, passenger_count=1, trip_distance=1.12, fare_amount=7.5, extra=0.0, mta_tax=0.5, tip_amount=1.66, tolls_amount=0.0, ehail_fee=None, improvement_surcharge=0.3, total_amount=9.96, payment_type=1, trip_type=1, congestion_surcharge=0.0),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2019, 1, 12, 5, 8, 28), lpep_dropoff_datetime=datetime.datetime(2019, 1, 12, 5, 8, 33), store_and_fwd_flag='N', RatecodeID=1, PULocationID=74, DOLocationID=74, passenger_count=1, trip_distance=0.03, fare_amount=2.5, extra=0.5, mta_tax=0.5, tip_amount=0.0, tolls_amount=0.0, ehail_fee=None, improvement_surcharge=0.3, total_amount=3.8, payment_type=2, trip_type=1, congestion_surcharge=None),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2019, 1, 27, 0, 55, 22), lpep_dropoff_date

In [14]:
rdd = (
    df_green
    .select("lpep_pickup_datetime", "PULocationID", "total_amount")
    .rdd
)

In [15]:
rdd.filter(lambda row: True).take(1)

[Row(lpep_pickup_datetime=datetime.datetime(2019, 1, 29, 15, 36, 35), PULocationID=40, total_amount=9.96)]

In [16]:
from datetime import datetime

start = datetime(year=2020, month=1, day=1)

def filter_outliers(row):
    return row.lpep_pickup_datetime >= start

In [17]:
rows = rdd.take(10)
row = rows[0]
row

Row(lpep_pickup_datetime=datetime.datetime(2019, 1, 29, 15, 36, 35), PULocationID=40, total_amount=9.96)

In [18]:
def prepare_for_grouping(row):
    hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
    zone = row.PULocationID
    key = (hour, zone)

    amount = row.total_amount
    count = 1
    value = (amount, count)
    return (key, value)

In [19]:
def calculate_revenue(left_value, right_value):
    left_amount, left_count = left_value
    right_amount, right_count = right_value

    output_amount = left_amount + right_amount
    output_count = left_count + right_count

    return (output_amount, output_count)

In [20]:
(
    rdd
    .filter(filter_outliers)
    .map(prepare_for_grouping)
    .reduceByKey(calculate_revenue)
    .take(5)
)

[((datetime.datetime(2020, 1, 10, 20, 0), 74), (909.4199999999995, 62)),
 ((datetime.datetime(2020, 1, 17, 10, 0), 7), (289.2300000000001, 27)),
 ((datetime.datetime(2020, 1, 10, 12, 0), 41), (528.0900000000003, 46)),
 ((datetime.datetime(2020, 1, 16, 10, 0), 242), (242.41, 9)),
 ((datetime.datetime(2020, 1, 15, 9, 0), 75), (898.6599999999999, 60))]

In [21]:
def unwrap(row):
    return (row[0][0], row[0][1], row[1][0], row[1][1])

In [22]:
from collections import namedtuple

RevenueRow = namedtuple("RevenueRow", ["hour", "zone", "revenue", "count"])

def unwrap(row):
    return RevenueRow(
        hour=row[0][0], 
        zone=row[0][1], 
        revenue=row[1][0], 
        count=row[1][1],
    )

In [23]:
from pyspark.sql import types

result_schema = types.StructType([
    types.StructField("hour", types.TimestampType(), True),
    types.StructField("zone", types.IntegerType(), True),
    types.StructField("revenue", types.DoubleType(), True),
    types.StructField("count", types.IntegerType(), True),
])

df_result = (
    rdd
    .filter(filter_outliers)
    .map(prepare_for_grouping)
    .reduceByKey(calculate_revenue)
    .map(unwrap)
    .toDF(result_schema)
)

In [24]:
columns = ['VendorID', 'lpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance']

(
    df_green 
    .select(columns)
    .show()
)

+--------+--------------------+------------+------------+-------------+
|VendorID|lpep_pickup_datetime|PULocationID|DOLocationID|trip_distance|
+--------+--------------------+------------+------------+-------------+
|       2| 2019-01-29 15:36:35|          40|         181|         1.12|
|       2| 2019-01-12 05:08:28|          74|          74|         0.03|
|       2| 2019-01-27 00:55:22|         255|          36|         3.38|
|       1| 2019-01-31 12:43:05|         166|         163|          3.1|
|       2| 2019-01-20 00:36:35|         264|          56|          0.0|
|       1| 2019-01-16 14:42:12|          22|          14|          2.8|
|       2| 2019-01-17 18:22:17|         181|          33|         2.38|
|       2| 2019-01-06 09:41:30|          41|         142|         3.23|
|       1| 2019-01-13 13:17:14|         173|         260|          2.3|
|       1| 2019-01-30 21:03:20|         189|         257|          1.9|
|       2| 2019-01-29 14:53:59|         228|         165|       

In [25]:
duration_rdd = (
    df_green
    .select(columns)
    .rdd
)

In [28]:
def apply_model_in_batch(partition):
    return [1]

duration_rdd.mapPartitions(apply_model_in_batch).collect()

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

In [29]:
def apply_model_in_batch(partition):
    cnt = 0
    for row in partition:
        cnt += 1
    return [cnt]

duration_rdd.mapPartitions(apply_model_in_batch).collect()

[1375942,
 1324820,
 1185775,
 1129411,
 1140044,
 981646,
 355784,
 342131,
 308138,
 204876]

In [ ]:
import pandas as pd

def apply_model_in_batch(rows):
    df = pd.DataFrame(rows, columns=columns)
    cnt = len(df)
    return [cnt]

26/10/27 11:56:45 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 230559 ms exceeds timeout 120000 ms
26/10/27 11:56:45 WARN SparkContext: Killing executors is not supported by current scheduler.
26/10/27 11:56:48 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stora